## Read Silver

In [0]:

from pyspark.sql.functions import (
    col,
    sum,
    round,
    when,
    lit
)

silver_table = "ev_registration.silver.silver_vehicle_registrations"

silver_df = spark.read.table(silver_table)

display(silver_df)


registration_year,maker,state,rto_code,rto,vehicle_category,vehicle_model,fuel,vehicle_class,sale_type,vehicle_count,ingestion_timestamp,registration_month,registration_month_number,is_ev,ev_type
2023,HYUNDAI MOTOR INDIA LTD,Punjab,PB19,RTO BARNALA,LIGHT MOTOR VEHICLE,CRETA 1.5 CRDI MT EX,DIESEL,Motor Car,Resale,1,2026-10-06T17:16:18.405Z,Mar,3,0,NON_EV
2022,TVS MOTOR COMPANY LTD,Odisha,OR26,NUAPADA RTO,TWO WHEELER(NT),TVS RADEON,PETROL,M-Cycle/Scooter,New,3,2026-10-06T17:16:18.405Z,Mar,3,0,NON_EV
2020,TOYOTA KIRLOSKAR MOTOR PVT LTD,Karnataka,KA63,DHARWAD EAST RTO,LIGHT MOTOR VEHICLE,TOYOTA URBAN CRUISER (HIGH MT),PETROL,Motor Car,New,2,2026-10-06T17:16:18.405Z,Dec,12,0,NON_EV
2022,MARUTI SUZUKI INDIA LTD,Chhattisgarh,CG14,Jashpur DTO,LIGHT MOTOR VEHICLE,SWIFT LXI,PETROL,Motor Car,New,1,2026-10-06T17:16:18.405Z,Sep,9,0,NON_EV
2023,TATA MOTORS PASSENGER VEHICLES LTD,Karnataka,KA51,ELECTRONIC CITY RTO,LIGHT MOTOR VEHICLE,HARRIER XZ+ 2LBS6PH2 #DRK,DIESEL,Motor Car,New,1,2026-10-06T17:16:18.405Z,Sep,9,0,NON_EV
2021,HERO ELECTRIC VEHICLES PVT. LTD,Rajasthan,RJ14,JAIPUR (FIRST) RTO,TWO WHEELER(NT),OPTIMA HS 500 ER,ELECTRIC(BOV),M-Cycle/Scooter,Resale,1,2026-10-06T17:16:18.405Z,Jan,1,1,BATTERY_EV
2021,BAJAJ AUTO LTD,Maharashtra,MH40,NAGPUR (RURAL),THREE WHEELER(T),BAJAJ MAXIMA C,DIESEL,Three Wheeler (Goods),Resale,1,2026-10-06T17:16:18.405Z,Jan,1,0,NON_EV
2020,SUZUKI MOTORCYCLE INDIA PVT LTD,Maharashtra,MH28,BULDHANA,TWO WHEELER(NT),ACCESS 125 DISC(ALLOY WHEEL),PETROL,M-Cycle/Scooter,Resale,1,2026-10-06T17:16:18.405Z,Oct,10,0,NON_EV
2023,HONDA MOTORCYCLE AND SCOOTER INDIA (P) LTD,Bihar,BR39,KATIHAR,TWO WHEELER(NT),ACTIVA SMART,PETROL,M-Cycle/Scooter,New,1,2026-10-06T17:16:18.405Z,Sep,9,0,NON_EV
2023,TVS MOTOR COMPANY LTD,Arunachal Pradesh,AR01,ITANAGAR CAPITAL REGION,TWO WHEELER(NT),TVS NTORQ 125 XT,PETROL,M-Cycle/Scooter,New,14,2026-10-06T17:16:18.405Z,Oct,10,0,NON_EV


## Aggregate By Vehicle Category

In [0]:
gold_vehicle_category_df = (
    silver_df
    .groupBy("vehicle_category")
    .agg(
        sum("vehicle_count").alias("total_registrations"),
        sum(
            when(
                col("is_ev") == 1, col("vehicle_count")
            ).otherwise(0)
        ).alias("ev_registrations"),
    )
)


## Calculate Non-EV Registrations

In [0]:
gold_vehicle_category_df = (
    gold_vehicle_category_df
    .withColumn(
        "non_ev_registration",
        col("total_registrations") - col("ev_registrations")
    )
)

##Calculate EV Penetration %

In [0]:
gold_vehicle_category_df = (
    gold_vehicle_category_df
    .withColumn(
        "ev_penetration_pct",
        round(
            col("ev_registrations") /
            col("total_registrations") * 100,
            2
        )
    )
)

## Calculate EV Market Share %

In [0]:
total_ev = (
    silver_df
    .filter(col("is_ev") == 1)
    .agg(sum("vehicle_count").alias("total_ev"))
    .collect()[0]["total_ev"]
)

gold_vehicle_category_df = (
    gold_vehicle_category_df
    .withColumn(
        "ev_market_share_pct",
        round(
            col("ev_registrations") / lit(total_ev) * 100,
            2
        )
    )
)

## Final Gold DataFrame

In [0]:
final_gold_vehicle_category_df = (
    gold_vehicle_category_df
    .select(
        "vehicle_category",
        "total_registrations",
        "ev_registrations",
        "non_ev_registration",
        "ev_penetration_pct",
        "ev_market_share_pct"
    )
    .orderBy(col("ev_registrations").desc())
)

print("Gold vehicle category records:", final_gold_vehicle_category_df.count())

final_gold_vehicle_category_df.printSchema()

display(final_gold_vehicle_category_df)

Gold vehicle category records: 16
root
 |-- vehicle_category: string (nullable = true)
 |-- total_registrations: long (nullable = true)
 |-- ev_registrations: long (nullable = true)
 |-- non_ev_registration: long (nullable = true)
 |-- ev_penetration_pct: double (nullable = true)
 |-- ev_market_share_pct: double (nullable = true)



vehicle_category,total_registrations,ev_registrations,non_ev_registration,ev_penetration_pct,ev_market_share_pct
TWO WHEELER(NT),2062297,82283,1980014,3.99,57.19
THREE WHEELER(T),96834,51846,44988,53.54,36.04
LIGHT MOTOR VEHICLE,512443,7932,504511,1.55,5.51
TWO WHEELER(T),1712,500,1212,29.21,0.35
HEAVY PASSENGER VEHICLE,2146,457,1689,21.3,0.32
LIGHT GOODS VEHICLE,65004,400,64604,0.62,0.28
LIGHT PASSENGER VEHICLE,24804,348,24456,1.4,0.24
OTHER THAN MENTIONED ABOVE,7434,63,7371,0.85,0.04
THREE WHEELER(NT),588,34,554,5.78,0.02
HEAVY GOODS VEHICLE,22817,4,22813,0.02,0.0


## Write Gold Table

In [0]:
gold_vehicle_category_table = (
    "ev_registration.gold.gold_ev_vehicle_category"
)

final_gold_vehicle_category_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable(gold_vehicle_category_table)

## Verify

In [0]:
gold_vehicle_category_check = spark.read.table(gold_vehicle_category_table)

display(
    gold_vehicle_category_check
    .orderBy(col("ev_registrations").desc())
)

vehicle_category,total_registrations,ev_registrations,non_ev_registration,ev_penetration_pct,ev_market_share_pct
TWO WHEELER(NT),2062297,82283,1980014,3.99,57.19
THREE WHEELER(T),96834,51846,44988,53.54,36.04
LIGHT MOTOR VEHICLE,512443,7932,504511,1.55,5.51
TWO WHEELER(T),1712,500,1212,29.21,0.35
HEAVY PASSENGER VEHICLE,2146,457,1689,21.3,0.32
LIGHT GOODS VEHICLE,65004,400,64604,0.62,0.28
LIGHT PASSENGER VEHICLE,24804,348,24456,1.4,0.24
OTHER THAN MENTIONED ABOVE,7434,63,7371,0.85,0.04
THREE WHEELER(NT),588,34,554,5.78,0.02
HEAVY GOODS VEHICLE,22817,4,22813,0.02,0.0
